# Phase 1: Dataset Setup & Preprocessing Pipeline

In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

### 1. Data Loading & Splitting

In [ ]:
# Load image paths and labels from the training directory
dataset_dir = "/kaggle/input/brain-tumor-mri-dataset/Training"  # change this path if running on Colab/local
data = []

for subdir, _, files in os.walk(dataset_dir):
    label = os.path.basename(subdir)  
    for file in files:
        file_path = os.path.join(subdir, file)
        data.append([file_path, label])
                
df = pd.DataFrame(data, columns=['file_path', 'label'])

# 80/10/10 split (train, validation, test)
# First pull out 20% for validation + testing
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
# Split the remaining 20% equally into val and test sets
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

### 2. Medical Preprocessing (CLAHE + Bilateral Filtering)

In [ ]:
def medical_preprocessing(img):
    """Filters noise and enhances contrast on MRI slices."""
    # Convert image to uint8 for OpenCV compatibility
    img_uint8 = (img).astype(np.uint8) if np.max(img) > 1.0 else (img * 255).astype(np.uint8)
    
    # Smooth out noise but keep structural edges sharp
    img_filtered = cv2.bilateralFilter(img_uint8, d=9, sigmaColor=75, sigmaSpace=75)
    
    # Run CLAHE on the L-channel to boost contrast without over-amplifying noise
    lab = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2LAB)
    l_channel, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)
    limg = cv2.merge((cl, a, b))
    img_clahe = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    
    # Scale pixels back to [0, 1]
    return img_clahe / 255.0

### 3. Data Augmentation

In [ ]:
# Setup data generators with preprocessing and light augmentation for training
datagen_train = ImageDataGenerator(
    preprocessing_function=medical_preprocessing,
    rotation_range=10,           # small rotations are safer for medical images
    brightness_range=(0.8, 1.2), # handle variations in scanner contrast
    horizontal_flip=True
)

# No augmentations for validation and testing, just the preprocessing
datagen_test = ImageDataGenerator(preprocessing_function=medical_preprocessing)

target_size = (224, 224)
batch_size = 32

train_generator = datagen_train.flow_from_dataframe(train_df, x_col='file_path', y_col='label', target_size=target_size, batch_size=batch_size, seed=42)
validation_generator = datagen_test.flow_from_dataframe(val_df, x_col='file_path', y_col='label', target_size=target_size, batch_size=batch_size, seed=42)
test_generator = datagen_test.flow_from_dataframe(test_df, x_col='file_path', y_col='label', target_size=target_size, batch_size=batch_size, shuffle=False)

# Phase 2: Baseline Model & Quantitative Metrics

### 1. Class Imbalance Mitigation

In [ ]:
# Calculate class weights to handle training class imbalance
class_labels = list(train_generator.class_indices.keys())
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights_array))
print(f"Class weights: {class_weights}")

### 2. Baseline Model Architecture & Callbacks

In [ ]:
# Define our baseline model (using pre-trained ResNet-50 as the feature extractor)
base_model = ResNet50(include_top=False, weights="imagenet", input_shape=(224, 224, 3), pooling='avg')

model = Sequential([
    base_model,
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.25),
    Dense(4, activation='softmax')  # 4 classes in the dataset
])

model.compile(
    optimizer=AdamW(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')]
)

model.summary()

# Early stopping to prevent overfitting, and decay LR if learning stalls
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

### 3. Model Training

In [ ]:
# Train the baseline model
history = model.fit(
    train_generator,
    epochs=25,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr]
)

### 4. Performance Evaluation & Confusion Matrix

In [ ]:
print("\n--- Evaluating model performance on the test set ---")
model.evaluate(test_generator)

y_true = test_generator.classes 
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)  

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=class_labels))

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()